# 🏭 Interactive Warehouse Digital Twin

This notebook guides you through creating a custom warehouse:
1. **Define warehouse dimensions** - Enter your warehouse size
2. **Add products** - Input products one by one or in batch
3. **Visualize** - See your warehouse in 3D with all products

Each product gets a unique position and ID automatically!

## 📦 Setup & Imports

In [ ]:
# For Google Colab, uncomment the following:
# !git clone https://github.com/spacebeige/3d_mapping.git
# %cd 3d_mapping
# !pip install -r requirements.txt

import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Add parent directory to path if running locally
if Path('..').absolute() not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(Path('..').absolute()))

from main import WarehouseDigitalTwin
from models import Product, ZoneType, Position3D
import pandas as pd
import numpy as np

print("✅ Setup complete!")

## 📏 Step 1: Define Warehouse Dimensions

Enter the dimensions of your warehouse below:

In [ ]:
# Get warehouse dimensions from user
print("🏗️ WAREHOUSE CONFIGURATION")
print("=" * 50)

warehouse_name = input("Enter warehouse name (default: My Distribution Center): ").strip() or "My Distribution Center"
length = float(input("Enter warehouse length in meters (default: 100): ") or "100")
width = float(input("Enter warehouse width in meters (default: 80): ") or "80")
height = float(input("Enter warehouse height in meters (default: 10): ") or "10")
num_aisles = int(input("Enter number of aisles (default: 10): ") or "10")

print("\n📊 Warehouse Configuration:")
print(f"   Name: {warehouse_name}")
print(f"   Dimensions: {length}m × {width}m × {height}m")
print(f"   Aisles: {num_aisles}")
print(f"   Volume: {length * width * height:,.0f} m³")

# Create warehouse
twin = WarehouseDigitalTwin()
warehouse = twin.create_warehouse(
    name=warehouse_name,
    length=length,
    width=width,
    height=height,
    num_aisles=num_aisles,
    aisle_width=3.0,
    rack_height=height * 0.8
)

print("\n✅ Warehouse created successfully!")

## 📦 Step 2: Add Products

Now add products to your warehouse. Each product will be automatically assigned a unique position!

In [ ]:
# Product collection
products = []
product_counter = 1

# Helper function to assign unique positions
def assign_position(product_index, total_aisles, warehouse_length, warehouse_width, warehouse_height):
    """
    Assign a unique position to each product based on its index.
    Distributes products evenly across aisles, racks, and shelves.
    """
    # Calculate configuration based on warehouse dimensions
    # More racks for longer warehouses (1 rack per 5m of length)
    racks_per_aisle = max(10, int(warehouse_length / 5))
    
    # More shelves for taller warehouses (1 shelf per 1.6m of usable height)
    usable_height = warehouse_height * 0.8  # 80% of height is usable
    shelves_per_rack = max(3, int(usable_height / 1.6))
    
    # Products capacity per aisle
    products_per_aisle = racks_per_aisle * shelves_per_rack
    
    # Calculate which aisle, rack, and shelf
    aisle_num = (product_index // products_per_aisle) % total_aisles
    rack_num = (product_index % products_per_aisle) // shelves_per_rack
    shelf_num = product_index % shelves_per_rack
    
    # Calculate 3D position
    aisle_spacing = warehouse_width / (total_aisles + 1)
    x = aisle_spacing * (aisle_num + 1)
    
    rack_spacing = warehouse_length / racks_per_aisle
    y = rack_spacing * (rack_num + 0.5)
    
    shelf_height = usable_height / shelves_per_rack
    z = shelf_height * (shelf_num + 0.5)
    
    return Position3D(x=x, y=y, z=z)

print("📦 PRODUCT INPUT")
print("=" * 50)
print("Enter product details. Type 'done' when finished.\n")

while True:
    print(f"\n--- Product #{product_counter} ---")
    
    item_id = input(f"Product ID (default: PROD{product_counter:04d}): ").strip() or f"PROD{product_counter:04d}"
    
    # Check if user wants to finish
    if item_id.lower() == 'done':
        break
    
    category = input("Category (electronics/groceries/pharma/apparel/automotive): ").strip() or "electronics"
    description = input("Description: ").strip() or f"Product {product_counter}"
    try:
        stock_level = int(input("Stock level (default: 100): ") or "100")
        if stock_level < 0:
            print(f"⚠️ Stock level cannot be negative, using 0")
            stock_level = 0
        
        daily_demand = float(input("Daily demand (default: 10): ") or "10")
        if daily_demand < 0:
            print(f"⚠️ Daily demand cannot be negative, using 0")
            daily_demand = 0
    except ValueError as e:
        print(f"⚠️ Invalid number, using defaults: {e}")
        stock_level = 100
        daily_demand = 10
    
    
    # Assign unique position
    position = assign_position(
        product_counter - 1,
        num_aisles,
        length,
        width,
        height
    )
    
    # Determine zone based on demand (high demand = Zone A, low = Zone D)
    if daily_demand > 50:
        zone = ZoneType.A
    elif daily_demand > 20:
        zone = ZoneType.B
    elif daily_demand > 10:
        zone = ZoneType.C
    else:
        zone = ZoneType.D
    
    # Create product
    product = Product(
        item_id=item_id,
        category=category,
        description=description,
        stock_level=stock_level,
        daily_demand=daily_demand,
        profit_per_unit=50.0,
        holding_cost_per_unit_day=0.15,
        turnover_ratio=daily_demand / stock_level if stock_level > 0 else 0,
        size_score=3.0,
        predicted_zone=zone,
        position=position
    )
    
    products.append(product)
    
    print(f"\n✅ Product added!")
    print(f"   ID: {item_id}")
    print(f"   Position: ({position.x:.2f}, {position.y:.2f}, {position.z:.2f})")
    print(f"   Zone: {zone.value}")
    
    product_counter += 1
    
    # Ask if user wants to add more
    more = input("\nAdd another product? (y/n, default: y): ").strip().lower()
    if more == 'n':
        break

print(f"\n✅ Total products added: {len(products)}")

if len(products) == 0:
    print("\n⚠️ No products added. Adding sample products...")
    # Add some default products
    for i in range(5):
        position = assign_position(i, num_aisles, length, width, height)
        product = Product(
            item_id=f"PROD{i+1:04d}",
            category="electronics",
            description=f"Sample Product {i+1}",
            stock_level=100,
            daily_demand=15.0,
            profit_per_unit=50.0,
            holding_cost_per_unit_day=0.15,
            turnover_ratio=0.15,
            size_score=3.0,
            predicted_zone=ZoneType.B,
            position=position
        )
        products.append(product)
    print(f"✅ Added {len(products)} sample products")

## 📊 Step 3: View Product Summary

In [ ]:
# Create a summary DataFrame
product_data = []
for p in products:
    product_data.append({
        'Product ID': p.item_id,
        'Category': p.category,
        'Description': p.description,
        'Stock': p.stock_level,
        'Daily Demand': p.daily_demand,
        'Zone': p.predicted_zone.value,
        'Position X': f"{p.position.x:.2f}",
        'Position Y': f"{p.position.y:.2f}",
        'Position Z': f"{p.position.z:.2f}"
    })

df = pd.DataFrame(product_data)
print("\n📋 PRODUCT SUMMARY")
print("=" * 80)
print(df.to_string(index=False))

# Statistics
print(f"\n📊 STATISTICS")
print("=" * 50)
print(f"Total Products: {len(products)}")
print(f"Total Stock: {sum(p.stock_level for p in products):,} units")
print(f"Total Daily Demand: {sum(p.daily_demand for p in products):.1f} units/day")
print(f"\nZone Distribution:")
zone_counts = {}
for p in products:
    zone = p.predicted_zone.value
    zone_counts[zone] = zone_counts.get(zone, 0) + 1
for zone, count in sorted(zone_counts.items()):
    print(f"  Zone {zone}: {count} products")

## 🎨 Step 4: Generate 3D Visualization

In [ ]:
from viz.warehouse_3d import Warehouse3DVisualizer

print("🎨 Generating 3D warehouse visualization...")

# Create visualizer
visualizer = Warehouse3DVisualizer(warehouse)

# Generate visualization
fig = visualizer.create_visualization(
    products=products,
    show_products=True,
    max_products_display=len(products)  # Show all products
)

# Show in notebook
fig.show()

print("\n✅ Visualization generated!")

## 💾 Step 5: Save & Download

Save your warehouse and product data for later use.

In [ ]:
# Save 3D visualization
print("💾 Saving files...")

# Save warehouse visualization
fig.write_html("warehouse_3d.html")
print("✅ Saved: warehouse_3d.html")

# Save product data as CSV
df.to_csv("products.csv", index=False)
print("✅ Saved: products.csv")

# Create a simple dashboard
if len(products) > 0:
    # Convert products to DataFrame for the twin system
    products_df = pd.DataFrame([{
        'item_id': p.item_id,
        'category': p.category,
        'description': p.description,
        'stock_level': p.stock_level,
        'daily_demand': p.daily_demand
    } for p in products])
    
    twin.load_products(products_df)
    
    try:
        # Create dashboard
        dashboard = twin.create_dashboard()
        dashboard.write_html("dashboard.html")
        print("✅ Saved: dashboard.html")
    except Exception as e:
        print(f"⚠️ Dashboard creation skipped: {e}")
        import traceback
        traceback.print_exc()

print("\n📁 Files ready for download!")

## 📥 Download Files (Google Colab only)

In [ ]:
# For Google Colab - download files
try:
    from google.colab import files
    print("📥 Downloading files...")
    files.download("warehouse_3d.html")
    files.download("products.csv")
    try:
        files.download("dashboard.html")
    except Exception:
        pass
    print("✅ Files downloaded!")
except ImportError:
    print("ℹ️ Not running in Google Colab - files saved to current directory")
    print("   You can find them at:")
    print("   - warehouse_3d.html")
    print("   - products.csv")
    print("   - dashboard.html (if created)")

## 🎉 Summary

Congratulations! You've created your custom warehouse digital twin:

✅ Defined warehouse dimensions  
✅ Added products with unique IDs and positions  
✅ Generated 3D visualization  
✅ Saved all data for future use  

### What's Next?
- Open `warehouse_3d.html` to explore your warehouse in 3D
- Use `products.csv` to import your data into other systems
- Modify and re-run cells to add more products or change dimensions

### Tips:
- Each product automatically gets a unique position based on its order
- Products are distributed evenly across aisles and shelves
- High-demand products are assigned to Zone A (closest to exits)
- Low-demand products go to Zone D (longer-term storage)